First experment, described in real\science\experiment_1_celebrities\exprement_description_link.txt

Requirments to run: HF_TOKEN setted

In [67]:
import sys
from pathlib import Path

# repo root = .../VisulaiztionInfoFlowDemo  (three parents up from this notebook)
REPO_ROOT = Path.cwd().parents[2] if (Path.cwd().name == "experiment_1_celebrities") else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))


## 1. Data loading

### Load information to model

In [1]:
from pydantic import BaseModel
class RawSentenceData(BaseModel):
    sentence:str
    name_start_index:int
    name_end_index:int
    name:str
    

In [2]:
import json
import re

# Load a
with open(r"data\a_about_celebrities.json", "r", encoding="utf-8") as f:
    a_about_celebrities_dict = json.load(f)

a_about_celebrities: list[RawSentenceData] = []
for info in a_about_celebrities_dict:
    end_of_first_name_index = info["sentence"].find(" ", info["start_of_name_index"] + 1)
    end_of_last_name_index = re.search(r" |'s",info["sentence"][end_of_first_name_index+1:]).start() + end_of_first_name_index+1
    a_about_celebrities.append(
        RawSentenceData(
            sentence=info["sentence"],
            name_start_index=info["start_of_name_index"],
            name_end_index=end_of_last_name_index,
            name=info["sentence"][info["start_of_name_index"] : end_of_last_name_index],
        )
    )


# Load b
with open(r"data\b_mentions_celebrities.json", "r", encoding="utf-8") as f:
    b_mentions_celebrities_dict = json.load(f)

b_mentions_celebrities: list[RawSentenceData] = []
for info in b_mentions_celebrities_dict:
    end_of_first_name_index = info["sentence"].find(" ", info["start_of_name_index"] + 1)
    end_of_last_name_index = re.search(r" |'s",info["sentence"][end_of_first_name_index+1:]).start() + end_of_first_name_index+1
    b_mentions_celebrities.append(
        RawSentenceData(
            sentence=info["sentence"],
            name_start_index=info["start_of_name_index"],
            name_end_index=end_of_last_name_index,
            name=info["sentence"][info["start_of_name_index"] : end_of_last_name_index],
        )
    )

### Verify names indexes data

In [3]:
print("A data:")
for sentece_info in a_about_celebrities:
    print(f"Sentence: {sentece_info.sentence}| Name: {sentece_info.name}")
    
print("--------------------------------------------------------------")
print("B data")
for sentece_info in b_mentions_celebrities:
    print(f"Sentence: {sentece_info.sentence}| Name: {sentece_info.name}")

A data:
Sentence: Taylor Swift kicked off her world tour with a three-hour setlist spanning every era of her career.| Name: Taylor Swift
Sentence: Did you see that Dwayne Johnson posted a new workout video this morning?| Name: Dwayne Johnson
Sentence: In a statement released Thursday, Emma Watson announced her return to acting after a brief hiatus.| Name: Emma Watson
Sentence: Honestly, I think Ryan Reynolds is one of the funniest people in Hollywood right now.| Name: Ryan Reynolds
Sentence: Serena Williams retired from professional tennis after an illustrious career spanning over two decades.| Name: Serena Williams
Sentence: Breaking: Elon Musk's latest venture has sparked intense debate among industry analysts.| Name: Elon Musk
Sentence: My mom absolutely adores Tom Hanks — she's seen every single one of his movies.| Name: Tom Hanks
Sentence: Beyoncé Knowles surprised fans with an unannounced album drop late last night.| Name: Beyoncé Knowles
Sentence: According to sources close to t

### Load generated fake celebrities names

In [4]:
import random
with open(r"data\fake_celebrities_62_tokens.json", "r", encoding="utf-8") as f:
    alternative_names = json.load(f)
random.seed(1321)
random.shuffle(alternative_names)

### Show sums of token number are equal (model dependent)

#### Setup

In [5]:
from dotenv import load_dotenv
import os
load_dotenv(r"..\..\..\.env.local")
HF_TOKEN = os.getenv("HF_TOKEN")

In [20]:
MODEL = "meta-llama/Llama-3.1-8B"

In [96]:
from pathlib import Path
from api_checks.api_cache import ModelAPICache
from tqdm import tqdm


model_api_cache = ModelAPICache(cache_path=Path(".model_api_cache"),hf_token=HF_TOKEN)
model_calcutor = model_api_cache.get_infomration_calculator(MODEL)

#### Calculate sum tokens amount

In [8]:
def calc_sum_name_tokens_amount(raw_sentences_data:list[RawSentenceData])->int:
    names_total_tokens = 0
    for sentence_info in raw_sentences_data:
        tokens_num = len(model_calcutor.calc_tokens(sentence_info.name)) -1 #Dont include BOS
        names_total_tokens += tokens_num
    return names_total_tokens
total_names_tokens_a = calc_sum_name_tokens_amount(a_about_celebrities)
total_names_tokens_b = calc_sum_name_tokens_amount(b_mentions_celebrities)

total_fake_names_tokens = 0
for name in alternative_names:
    tokens_num = len(model_calcutor.calc_tokens(name)) -1 #Dont include BOS
    total_fake_names_tokens += tokens_num
    

In [9]:
assert total_names_tokens_a == total_names_tokens_b == total_fake_names_tokens
print(f"All total tokens number in names are equal to {total_names_tokens_a}")

All total tokens number in names are equal to 62


### Replace information in sentences

In [24]:
a_sentnces_fake_names = [raw_sentence_info.sentence.replace(raw_sentence_info.name,fake_name) for raw_sentence_info,fake_name in zip(a_about_celebrities,alternative_names)]
b_sentences_fake_names = [raw_sentence_info.sentence.replace(raw_sentence_info.name,fake_name) for raw_sentence_info,fake_name in zip(b_mentions_celebrities,alternative_names)]

In [ ]:
for i,sentence in enumerate(a_sentnces_fake_names):
    print(i,sentence)

In [ ]:
for i,sentence in enumerate(b_sentences_fake_names):
    print(i,sentence)

In [43]:
def fake_name_sentences_to_sentence_raw_data(
    sentences_with_fake_names: list[str], sentences_raw_data: list[RawSentenceData], fake_names: list[str]
) -> list[RawSentenceData]:
    result = []
    for fake_sentence, sentence_raw_data, fake_name in zip(sentences_with_fake_names, sentences_raw_data, fake_names):
        fake_sentence_raw_data = RawSentenceData(
            sentence=fake_sentence,
            name_start_index=sentence_raw_data.name_start_index,
            name_end_index=sentence_raw_data.name_start_index + len(fake_name),
            name=fake_name
        )
        result.append(fake_sentence_raw_data)
    return result
        
a_fake_names_raw_data = fake_name_sentences_to_sentence_raw_data(a_sentnces_fake_names,a_about_celebrities,alternative_names)
b_fake_names_raw_data = fake_name_sentences_to_sentence_raw_data(b_sentences_fake_names,b_mentions_celebrities,alternative_names)


### Orgnize sentence by tokens (model dependent)

In [77]:
class SentenceTokensInformation(BaseModel):
    sentence: str
    tokens: list[str]
    name_tokens_indexes: list[int]


def sentence_token_information_from_raw_data(raw_sentence_data: RawSentenceData, model_name: str):
    calculator = model_api_cache.get_infomration_calculator(model_name)
    sentence_tokens = calculator.calc_tokens(raw_sentence_data.sentence)

    current_character_index = 0
    current_token_index = 1  # skip BOS
    name_tokens_indexes = []
    while current_character_index < raw_sentence_data.name_start_index:
        current_character_index += len(sentence_tokens[current_token_index])
        current_token_index += 1
        
    while current_character_index<= raw_sentence_data.name_end_index:
        name_tokens_indexes.append(current_token_index)
        current_character_index += len(sentence_tokens[current_token_index])
        current_token_index += 1
        
    # In case token include token name
    return SentenceTokensInformation(sentence=raw_sentence_data.sentence,tokens=sentence_tokens,name_tokens_indexes=name_tokens_indexes)

In [78]:
a_tokens_info = [sentence_token_information_from_raw_data(sentence_raw_data,MODEL) for sentence_raw_data in a_about_celebrities]
a_fake_name_tokens_info = [sentence_token_information_from_raw_data(sentence_raw_data,MODEL) for sentence_raw_data in a_fake_names_raw_data]
b_tokens_info = [sentence_token_information_from_raw_data(sentence_raw_data,MODEL) for sentence_raw_data in b_mentions_celebrities]
b_fake_name_tokens_info = [sentence_token_information_from_raw_data(sentence_raw_data,MODEL) for sentence_raw_data in b_fake_names_raw_data]

In [79]:
a_tokens_info = (a_tokens_info,a_fake_name_tokens_info)
b_tokens_info = (b_tokens_info,b_fake_name_tokens_info)
print(a_tokens_info)

([SentenceTokensInformation(sentence='Taylor Swift kicked off her world tour with a three-hour setlist spanning every era of her career.', tokens=['<|begin_of_text|>', 'Taylor', ' Swift', ' kicked', ' off', ' her', ' world', ' tour', ' with', ' a', ' three', '-hour', ' set', 'list', ' spanning', ' every', ' era', ' of', ' her', ' career', '.'], name_tokens_indexes=[1, 2, 3]), SentenceTokensInformation(sentence='Did you see that Dwayne Johnson posted a new workout video this morning?', tokens=['<|begin_of_text|>', 'Did', ' you', ' see', ' that', ' D', 'wayne', ' Johnson', ' posted', ' a', ' new', ' workout', ' video', ' this', ' morning', '?'], name_tokens_indexes=[6, 7, 8]), SentenceTokensInformation(sentence='In a statement released Thursday, Emma Watson announced her return to acting after a brief hiatus.', tokens=['<|begin_of_text|>', 'In', ' a', ' statement', ' released', ' Thursday', ',', ' Emma', ' Watson', ' announced', ' her', ' return', ' to', ' acting', ' after', ' a', ' brie

### Check that same amount of tokens after name

In [80]:
def check_same(group_tokens_info:tuple[list[SentenceTokensInformation],list[SentenceTokensInformation]]):
    for (real_sent,fake_sent) in zip(group_tokens_info[0],group_tokens_info[1]):
        assert len(real_sent.tokens)-max(real_sent.name_tokens_indexes) == len(fake_sent.tokens)-max(fake_sent.name_tokens_indexes), f"Error, not same amount of tokens after name\n Real: {real_sent.tokens} | Fake: {fake_sent.tokens}"
        
check_same(a_tokens_info)
check_same(b_tokens_info)

## 2. load data

In [90]:
from pydantic_settings import SettingsConfigDict

from api_checks.full_run_result import Contributions, FullRunResults
import torch


class ContributionsNorms(BaseModel):
    post_attention_contributions_norms: torch.Tensor  # (layer,position (prompt_len_after_name),source (prompt_len))
    post_mlp_contributions_norms: torch.Tensor  # (layer,position (prompt_len_after_name),source (prompt_len))

    model_config = SettingsConfigDict(arbitrary_types_allowed=True)

class ExprimentPair(BaseModel):
    real_names: tuple[SentenceTokensInformation, ContributionsNorms]
    fake_names: tuple[SentenceTokensInformation, ContributionsNorms]


def contributions_to_contributions_norm(
    contributions: Contributions, norm_func: callable = lambda x: x.norm(dim=-1)
) -> ContributionsNorms:
    post_attention_contributions_norms = norm_func(contributions.post_attention_contribution)
    post_mlp_contributions_norms = norm_func(contributions.post_mlp_contribution)
    return ContributionsNorms(
        post_attention_contributions_norms=post_attention_contributions_norms,
        post_mlp_contributions_norms=post_mlp_contributions_norms,
    )
    
def strip_contributions_norms(contributions_norms:ContributionsNorms,rng:range)->ContributionsNorms:
    return ContributionsNorms(post_attention_contributions_norms=contributions_norms.post_attention_contributions_norms[rng],
                              post_mlp_contributions_norms=contributions_norms.post_mlp_contributions_norms[rng])


### Calculate the contributions and keep only the norms in the relavant positions (after name mentioned)

In [ ]:
# Note: If computation heavy, could try to cache information until name appers should be the same in both sentences

def calculate_expirment_info(
    group_tokens_info: tuple[list[SentenceTokensInformation], list[SentenceTokensInformation]],
) -> list[ExprimentPair]:
    experiment_group = []
    for real_sent, fake_sent in tqdm(zip(group_tokens_info[0], group_tokens_info[1])):
        def sentence_tokens_info_to_relevant_contributions_norms(sentence_info:SentenceTokensInformation)->ContributionsNorms:
            sent_full_run_result = model_api_cache.get_infomration_calculator(MODEL).calc(sentence_info.sentence)
            
            sent_contributions_norms = contributions_to_contributions_norm(sent_full_run_result.contributions)
            relevant_contributions_norms = strip_contributions_norms(
                sent_contributions_norms, range(max(real_sent.name_tokens_indexes) + 1, len(real_sent.tokens))
            )
            return relevant_contributions_norms
        relevant_real_names_contributions_norms = sentence_tokens_info_to_relevant_contributions_norms(real_sent)
        relevant_fake_names_contributions_norms = sentence_tokens_info_to_relevant_contributions_norms(fake_sent)

        expriment_pair = ExprimentPair(
            real_names=(real_sent, relevant_real_names_contributions_norms), fake_names=(fake_sent, relevant_fake_names_contributions_norms)
        )
        experiment_group.append(expriment_pair)

        experiment_group.append(expriment_pair)
    return experiment_group

In [97]:
print(sum([len(a.tokens) for a in a_tokens_info[0]])//20)

18
